# Protoplanetary disk simulation with limb-darkened disks

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/distributed_applications_tutorials/simulation/protoplanetary_disk_simulation.ipynb)

Simulate an ALMA Band 6 observation of a **protoplanetary disk** modelled as an inclined
**limb-darkened disk** plus a point source for the central star, verify the simulated
visibilities against the analytic disk model, image the data with AstroVIPER and compare the
restored image with the model convolved by the clean beam, first with Hogbom CLEAN and then
with the Adaptive Scale Pixel (ASP) deconvolver, whose Gaussian components suit a smooth disk.

---
## The limb-darkened disk component

Besides point sources and Gaussians the simulator can place **limb-darkened elliptical disks**
on the sky.  The brightness follows the power-law limb-darkening law of
[Hestroffer (1997)](https://ui.adsabs.harvard.edu/abs/1997A%26A...327..199H):

$$ I(\mu) = I_0\,\mu^{\alpha}, \qquad \mu = \sqrt{1 - (2 r / D)^2}, \qquad r < D/2 $$

where $D$ is the outer diameter and $\alpha$ the limb-darkening exponent:

| $\alpha$ | sky brightness | normalised visibility |
| --- | --- | --- |
| $0$ | uniform disk | $2 J_1(x)/x$ |
| $> 0$ | darker towards the limb (stellar photospheres, smooth dust disks) | $\Gamma(\nu+1)\,(2/x)^{\nu} J_{\nu}(x)$, $\nu = \alpha/2 + 1$ |
| $-1$ | optically thin spherical shell (limb brightened) | $\sin x / x$ |
| $-2$ | infinitely thin ring (all flux on the rim, e.g. a black-hole photon ring) | $J_0(x)$ |

with $x = \pi D q$ for a baseline of length $q$ wavelengths.  An **inclined** disk is the same
profile compressed along its minor axis: it is described by ``[major, minor, position angle]``
exactly like a Gaussian source (``minor = major cos i`` for a circular disk of inclination $i$),
and its visibility is the circular formula evaluated at the *deprojected* baseline

$$ x = \pi \sqrt{\,\mathrm{major}^2 (u \sin\mathrm{pa} + v \cos\mathrm{pa})^2 + \mathrm{minor}^2 (u \cos\mathrm{pa} - v \sin\mathrm{pa})^2 }. $$

Like a Gaussian, a disk is simulated as a point source whose visibilities are multiplied by this
analytic response (``limb_darkened_disk_uv_response``); its image-plane twin
``limb_darkened_disk_image`` is used below to build the expected restored image.  The antenna
beams are sampled at the disk centre, which is accurate for disks much smaller than the primary
beam (here 2.4'' against ALMA's 27'' Band 6 beam).

---
## API

In [ ]:
from astroviper.distributed_applications.simulation import simulate_processing_set

simulate_processing_set?

## Install AstroVIPER

In [ ]:
import os
from importlib.metadata import version

try:
    import astroviper  # noqa: F401

    print("Using astroviper version", version("astroviper"))
except ImportError:
    os.system("pip install --upgrade astroviper")
    import astroviper  # noqa: F401

    print("Installed astroviper version", version("astroviper"))

In [ ]:
# Set True for interactive (zoom / pan) plots via the ipympl widget backend
# (``pip install ipympl``).  Keep False for the automated notebook tests, which
# execute headless.
INTERACTIVE_PLOTS = False

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from astropy.coordinates import SkyCoord
from IPython import get_ipython

get_ipython().run_line_magic("matplotlib", "widget" if INTERACTIVE_PLOTS else "inline")

xr.set_options(display_style="html")
ARCSEC_TO_RAD = np.pi / (180 * 3600)

In [ ]:
from toolviper.dask.client import local_client

viper_client = local_client(cores=4, memory_limit="4GB")
viper_client

## Brightness profiles and visibility curves of the model family

``limb_darkened_disk_image`` and ``limb_darkened_disk_uv_response`` are the image-plane and
visibility forms of the same unit-flux disk.  Left: radial brightness for several exponents
(the shell diverges integrably at the rim; the ring is not drawn).  Right: the normalised
visibility against $x = \pi D q$: limb darkening pushes the first null outwards and lowers the
side lobes, limb brightening does the opposite, and the thin ring never decays.

In [ ]:
from astroviper.processing_functions.simulation import (
    limb_darkened_disk_image,
    limb_darkened_disk_uv_response,
)

diameter = 1.0 * ARCSEC_TO_RAD  # any size: the curves are plotted in scaled units
r = np.linspace(0, 0.5, 400) * diameter
x = np.linspace(0, 20, 800)
q = x / (np.pi * diameter)
profiles = {
    r"$\alpha=2$": 2.0,
    r"$\alpha=1$": 1.0,
    r"$\alpha=0$ (uniform)": 0.0,
    r"$\alpha=-1$ (shell)": -1.0,
}

fig, (ax_sky, ax_uv) = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for label, alpha in profiles.items():
    brightness = limb_darkened_disk_image(r, 0.0, diameter, diameter, 0.0, alpha)
    ax_sky.plot(2 * r / diameter, brightness / brightness[0], label=label)
    ax_uv.plot(
        x,
        limb_darkened_disk_uv_response(q, 0.0, diameter, diameter, 0.0, alpha),
        label=label,
    )
ax_uv.plot(
    x,
    limb_darkened_disk_uv_response(q, 0.0, diameter, diameter, 0.0, -2.0),
    "k--",
    label=r"$\alpha=-2$ (ring)",
)
ax_sky.set_xlabel("2 r / D")
ax_sky.set_ylabel("brightness / central brightness")
ax_sky.set_ylim(0, 3)
ax_sky.set_title("radial brightness profile")
ax_sky.legend()
ax_uv.axhline(0, color="gray", lw=0.8)
ax_uv.set_xlabel(r"$x = \pi D q$")
ax_uv.set_ylabel("normalised visibility")
ax_uv.set_title("visibility of a unit-flux disk")
ax_uv.legend()
plt.show()

## ALMA 12 m array, configuration C-4

The Cycle 8 C-4 configuration (43 antennas, baselines up to 784 m) resolves a 2.4'' disk about
seven times at 230 GHz while its shortest baselines still recover the total flux.

In [ ]:
from astroviper.utils.telescope_layout import read_telescope_layout

antenna_xds = read_telescope_layout("alma.cycle8.4")
n_antenna = antenna_xds.sizes["antenna_name"]
positions = antenna_xds.ANTENNA_POSITION.values
baseline_length = np.linalg.norm(positions[:, None, :] - positions[None, :, :], axis=-1)
baseline_length = baseline_length[np.triu_indices(n_antenna, 1)]
frequency_hz = 230e9
wavelength = 299792458.0 / frequency_hz
print(
    f"{n_antenna} antennas, {len(baseline_length)} baselines from {baseline_length.min():.0f} to {baseline_length.max():.0f} m"
)
print(
    f"approximate resolution at {frequency_hz / 1e9:.0f} GHz: {wavelength / baseline_length.max() / ARCSEC_TO_RAD:.2f} arcsec"
)

## The sky: an inclined protoplanetary disk and its star

An HL Tau-like disk at 140 pc: outer diameter 2.4'' (170 au), inclination 40 degrees, position
angle 120 degrees, integrated flux 0.5 Jy at 230 GHz and a limb-darkening exponent
$\alpha = 1.5$, i.e. a brightness $\propto (1 - (2r/D)^2)^{0.75}$ that falls smoothly to zero at the
outer edge.  The star's free-free emission is a 3 mJy point source at the disk centre.  Both are
placed at the phase centre.  (Thermal noise is left off so that the checks below are exact; see
the noise tutorial for ``noise_params``.)

In [ ]:
phase_center = SkyCoord(ra="04h31m38.4s", dec="+18d13m57.7s", frame="icrs")
phase_center_ra_dec = np.array([phase_center.ra.rad, phase_center.dec.rad])[None, :]

flux_disk = 0.5  # Jy, integrated
major = 2.4 * ARCSEC_TO_RAD  # outer diameter
inclination = np.deg2rad(40.0)
minor = major * np.cos(inclination)
position_angle = np.deg2rad(120.0)
limb_darkening = 1.5
flux_star = 0.003  # Jy

disk_source_flux = np.array([flux_disk, 0, 0, flux_disk])[None, None, None, :]
disk_source_ra_dec = phase_center_ra_dec[None, :, :]
disk_source_shape = np.array([[major, minor, position_angle]])
disk_source_limb_darkening = np.array([limb_darkening])
point_source_flux = np.array([flux_star, 0, 0, flux_star])[None, None, None, :]
point_source_ra_dec = phase_center_ra_dec[None, :, :]

first_null_x = 5.0  # first zero of J_nu for nu = alpha/2 + 1 = 1.75
print(
    f"minor axis {minor / ARCSEC_TO_RAD:.2f} arcsec; first visibility null along the major axis at "
    f"~{first_null_x / (np.pi * major) * wavelength:.0f} m ({first_null_x / (np.pi * major) / 1e3:.0f} k-lambda)"
)

## Simulate

The disk enters through ``disk_source_flux``, ``disk_source_ra_dec``, ``disk_source_shape`` and
``disk_source_limb_darkening``; the star through the point-source arguments.  Two hours centred
on transit, two 2 GHz channels.

In [ ]:
from astroviper.utils.beam_models import airy_disk_model

result = simulate_processing_set(
    ps_store="disk_sim.ps.zarr",
    antenna_xds=antenna_xds,
    time_params={
        "time_start": "2021-10-03T07:15:00.000",
        "time_delta": 600.0,
        "n_samples": 12,
    },
    frequency_params={
        "freq_start": frequency_hz,
        "freq_delta": 2e9,
        "n_channels": 2,
        "channel_width": 2e9,
        "spectral_window_name": "Band6",
    },
    polarization=["XX", "YY"],
    point_source_flux=point_source_flux,
    point_source_ra_dec=point_source_ra_dec,
    disk_source_flux=disk_source_flux,
    disk_source_ra_dec=disk_source_ra_dec,
    disk_source_shape=disk_source_shape,
    disk_source_limb_darkening=disk_source_limb_darkening,
    phase_center_ra_dec=phase_center_ra_dec,
    beam_models=[airy_disk_model("alma")],
    beam_model_map=np.zeros(n_antenna, dtype=int),
    n_time_chunks=2,
    n_frequency_chunks=2,
    overwrite=True,
)
result["timing_node_tasks"][["task_id", "T_uvw", "T_visibilities", "T_write"]]

## Visibilities against the deprojected baseline

Rotating the $(u, v)$ coordinates to the disk's position angle and compressing the minor-axis
component by ``minor / major`` *deprojects* the inclined disk into a face-on one, so every
visibility falls on the one-dimensional analytic curve
$F_{\mathrm{disk}}\,V(x) + F_{\mathrm{star}}$.  Both sources sit at the phase centre, so the
visibilities are real up to the tiny $w$ term of the extended disk (the simulator keeps it; the
next cell shows it is $10^{-5}$ of the flux here): the real part is plotted, which also shows the
negative lobe beyond the first null.  Against the *projected* baseline length (right) the same points scatter, the
signature of an inclined source.

In [ ]:
from xradio.measurement_set import open_processing_set

ps_xdt = open_processing_set("disk_sim.ps.zarr")
ms_xds = ps_xdt[result["ms_name"]].ds
uvw = ms_xds.UVW.values  # [time, baseline, 3] metres
visibility = ms_xds.VISIBILITY.sel(
    polarization="XX"
).values  # [time, baseline, frequency]
u = uvw[:, :, 0, None] * ms_xds.frequency.values / 299792458.0  # wavelengths
v = uvw[:, :, 1, None] * ms_xds.frequency.values / 299792458.0
w = uvw[:, :, 2, None] * ms_xds.frequency.values / 299792458.0

u_major = u * np.sin(position_angle) + v * np.cos(position_angle)
u_minor = u * np.cos(position_angle) - v * np.sin(position_angle)
deprojected = np.sqrt(u_major**2 + (minor / major * u_minor) ** 2)
projected = np.sqrt(u**2 + v**2)


def model_curve(q):
    """Analytic disk + star for a face-on (deprojected) baseline length q [wavelengths]."""
    return (
        flux_disk
        * limb_darkened_disk_uv_response(q, 0.0, major, major, 0.0, limb_darkening)
        + flux_star
    )


expected = (
    flux_disk
    * limb_darkened_disk_uv_response(
        u, v, major, minor, position_angle, limb_darkening, w=w
    )
    + flux_star
)
print(
    "max |simulated - analytic (with the w term)| =",
    np.abs(visibility - expected).max(),
    "Jy",
)
print(
    "max |with - without the w term| =",
    np.abs(
        expected
        - flux_disk
        * limb_darkened_disk_uv_response(
            u, v, major, minor, position_angle, limb_darkening
        )
        - flux_star
    ).max(),
    "Jy (negligible here: pi w R^2 ~ 1e-4)",
)

q_curve = np.linspace(0, projected.max(), 600)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True, sharey=True)
for ax, baseline, title in [
    (axes[0], deprojected, "deprojected baseline"),
    (axes[1], projected, "projected baseline"),
]:
    ax.plot(
        baseline.ravel() / 1e3,
        visibility.real.ravel(),
        ".",
        ms=3,
        alpha=0.4,
        label="simulated (XX)",
    )
    ax.plot(
        q_curve / 1e3, model_curve(q_curve), "r-", lw=1.2, label="analytic disk + star"
    )
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_xlabel(f"{title} [k$\\lambda$]")
    ax.set_title(title)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Re V [Jy]")
axes[0].legend()
plt.show()

## Dirty and cleaned images

Briggs weighting, a 0.05'' cell and Hogbom CLEAN down to 0.1 mJy/beam.  The restored image is
primary-beam corrected (``primary_beam_correction=True``).

In [ ]:
from xradio.image import load_image, make_empty_sky_image

from astroviper.distributed_applications.imaging import image_cube_single_field

image_size = np.array([256, 256])
cell_size_arcsec = 0.05
cell_size = np.array([-cell_size_arcsec, cell_size_arcsec]) * ARCSEC_TO_RAD


def image_simulation(
    ps_store, image_store, niter=0, n_chunks=2, deconvolver="hogbom_many_threads"
):
    """Make a (dirty or cleaned) cube of a simulated processing set with AstroVIPER.

    Returns the image dataset and the imager's result dict (per-chunk timings
    and the per-plane deconvolution statistics).
    """
    ps_xdt = open_processing_set(ps_store)
    combined = ps_xdt.xr_ps.get_combined_field_and_source_xds()
    phase_direction = combined.FIELD_PHASE_CENTER_DIRECTION.sel(
        field_name=combined.attrs["center_field_name"]
    ).values
    keep = [
        "sky_residual",
        "point_spread_function",
        "primary_beam",
        "beam_fit_params_point_spread_function",
    ]
    if niter > 0:
        keep += ["sky_model", "mask"]
    result = image_cube_single_field(
        ps_store=ps_store,
        image_store=image_store,
        image_params={
            "image_size": list(image_size),
            "cell_size": cell_size,
            "phase_direction": phase_direction,
            "frequency_coords": ps_xdt.xr_ps.get_freq_axis().values,
            "polarization_coords": ["I"],
            "time_coords": [0],
            "fft_padding": 1.2,
            "cpp_gridder": True,
        },
        imaging_weights_params={
            "weighting": "briggs",
            "robust": 0.5,
            "casa_weighting_implementation": True,
        },
        iteration_control_params={
            "max_iter": niter,
            "max_cycles": -1 if niter > 0 else 0,
            "threshold": 1e-4,
            "gain": 0.1,
            "psf_sidelobe_factor": 1.5,
            "max_iter_per_cycle": -1,
            "min_psf_fraction": 0.05,
            "max_psf_fraction": 0.8,
            "primary_beam_limit": 0.2,
        },
        gridder="prolate_spheroidal",
        deconvolver=deconvolver,
        scan_intents="OBSERVE_TARGET#ON_SOURCE",
        image_data_variables_keep=keep,
        processing_set_data_group_name="base",
        single_precision_image=False,
        processing_function_threads=1,
        n_mapping_parallelism={"frequency": n_chunks},
        overwrite=True,
        restore=niter > 0,
        primary_beam_correction=niter > 0,
    )
    return add_sky_coordinates(load_image(image_store)), result


def add_sky_coordinates(img_xds):
    """Attach 2-D right_ascension / declination pixel coordinates (SIN grid)."""
    img_xds.attrs["type"] = "image_dataset"  # the xr_img accessor checks this
    reference_direction = img_xds.attrs["coordinate_system_info"][
        "reference_direction"
    ]["data"]
    template = make_empty_sky_image(
        phase_center=np.asarray(reference_direction),
        image_size=[img_xds.sizes["l"], img_xds.sizes["m"]],
        cell_size=img_xds.xr_img.get_lm_cell_size(),
        frequency_coords=img_xds.frequency.values,
        pol_coords=list(img_xds.polarization.values),
        time_coords=list(img_xds.time.values),
        do_sky_coords=True,
    )
    # assign by raw values: aligning on the template's float l / m coords
    # would reindex the sky coordinates to NaN
    return img_xds.assign_coords(
        right_ascension=(("l", "m"), template.right_ascension.values),
        declination=(("l", "m"), template.declination.values),
    )


dirty, _ = image_simulation("disk_sim.ps.zarr", "disk_sim_dirty.img.zarr", niter=0)
clean, hogbom_result = image_simulation(
    "disk_sim.ps.zarr", "disk_sim_clean.img.zarr", niter=5000
)
beam_fit = clean.BEAM_FIT_PARAMS_POINT_SPREAD_FUNCTION.isel(
    time=0, frequency=0, polarization=0
).values
print(
    f"clean beam: {beam_fit[0] / ARCSEC_TO_RAD:.3f} x {beam_fit[1] / ARCSEC_TO_RAD:.3f} arcsec, "
    f"pa {np.rad2deg(beam_fit[2]):.1f} deg"
)

## All imaging data products

Image pixels on the bottom / left axes, right ascension / declination on the top / right.  The
inclined disk is the tilted ellipse in ``SKY_RESTORED``; ``SKY_MODEL`` holds the CLEAN components.

In [ ]:
colorbar_labels = {
    "SKY_RESTORED": "Jy/beam",
    "SKY_RESTORED_PRIMARY_BEAM_CORRECTED": "Jy/beam",
    "SKY_RESIDUAL": "Jy/beam",
    "SKY_MODEL": "Jy/pixel",
    "POINT_SPREAD_FUNCTION": "response",
    "PRIMARY_BEAM": "power response",
    "MASK": "boolean",
}


def _sky_coordinate_axes(ax, img_xds):
    from astropy import units as u
    from astropy.coordinates import Angle

    ra = img_xds.right_ascension.values
    dec = img_xds.declination.values
    n_l, n_m = ra.shape
    top = ax.secondary_xaxis("top")
    top.set_xticks(np.linspace(0, n_l - 1, 3))
    top.set_xticklabels(
        [
            Angle(ra[int(pix), n_m // 2], u.rad).to_string(
                unit=u.hourangle, precision=1
            )
            for pix in np.linspace(0, n_l - 1, 3)
        ],
        fontsize=7,
    )
    top.set_xlabel("right ascension", fontsize=8)
    right = ax.secondary_yaxis("right")
    right.set_yticks(np.linspace(0, n_m - 1, 3))
    right.set_yticklabels(
        [
            Angle(dec[n_l // 2, int(pix)], u.rad).to_string(unit=u.deg, precision=0)
            for pix in np.linspace(0, n_m - 1, 3)
        ],
        fontsize=7,
    )
    right.set_ylabel("declination", fontsize=8)


def show_all_products(img_xds, suptitle, frequency=0, polarization=0, n_cols=4):
    """One-figure montage of every image-plane data product."""
    variables = [
        name for name in img_xds.data_vars if {"l", "m"} <= set(img_xds[name].dims)
    ]
    n_rows = -(-len(variables) // n_cols)
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(4.6 * n_cols, 4.3 * n_rows), constrained_layout=True
    )
    for ax in np.ravel(axes):
        ax.set_axis_off()
    for ax, variable in zip(np.ravel(axes), variables, strict=False):
        ax.set_axis_on()
        plane = (
            img_xds[variable]
            .isel(time=0, frequency=frequency, polarization=polarization)
            .values
        )
        im = ax.imshow(plane.T, origin="lower", cmap="viridis")
        ax.set_title(variable, fontsize=10)
        ax.set_xlabel("l [pixel]")
        ax.set_ylabel("m [pixel]")
        _sky_coordinate_axes(ax, img_xds)
        fig.colorbar(im, ax=ax, shrink=0.75, label=colorbar_labels.get(variable, ""))
    fig.suptitle(
        f"{suptitle} (channel {frequency}, {img_xds.frequency.values[frequency] / 1e9:.2f} GHz)"
    )
    return fig


show_all_products(clean, "protoplanetary disk + star, Hogbom CLEAN")
plt.show()

## Restored image versus the analytic model

The expected restored image is the input sky (``limb_darkened_disk_image`` in Jy/pixel plus the
star) convolved with the fitted clean beam, using the restore step's own beam kernel.  Cuts
through the disk centre along the major and minor axes show the inclination (the minor-axis cut
is narrower) and the smooth limb-darkened edge.  The table checks the integrated flux
(sum x pixel area / beam area) and the peak.

In [ ]:
import scipy.fft

from astroviper.processing_functions.imaging.restore import _elliptical_gaussian_kernel

restored = clean.SKY_RESTORED_PRIMARY_BEAM_CORRECTED.isel(
    time=0, frequency=0, polarization=0
).values
l_grid, m_grid = np.meshgrid(clean.l.values, clean.m.values, indexing="ij")
pixel_area = float(np.abs(cell_size[0] * cell_size[1]))
model = (
    flux_disk
    * limb_darkened_disk_image(
        l_grid, m_grid, major, minor, position_angle, limb_darkening
    )
    * pixel_area
)
centre = tuple(image_size // 2)
model[centre] += flux_star
pixel = cell_size_arcsec * ARCSEC_TO_RAD
kernel = _elliptical_gaussian_kernel(
    *image_size, beam_fit[0] / pixel, beam_fit[1] / pixel, beam_fit[2], np.float64
)
expected_restored = scipy.fft.irfft2(
    scipy.fft.rfft2(model) * scipy.fft.rfft2(scipy.fft.ifftshift(kernel)),
    s=tuple(image_size),
)
difference = restored - expected_restored


def flux_table(restored, difference):
    """Integrated flux, peak and model deviation of a primary-beam-corrected restored image."""
    beam_area = np.pi / (4 * np.log(2)) * beam_fit[0] * beam_fit[1]
    return pd.DataFrame(
        [
            {
                "quantity": "integrated flux [Jy]",
                "restored": np.nansum(restored) * pixel_area / beam_area,
                "expected": flux_disk + flux_star,
            },
            {
                "quantity": "peak [Jy/beam]",
                "restored": np.nanmax(restored),
                "expected": expected_restored.max(),
            },
            {
                "quantity": "max |restored - expected| / peak",
                "restored": np.nanmax(np.abs(difference)) / expected_restored.max(),
                "expected": 0.0,
            },
        ]
    ).round(5)


extent_arcsec = (
    np.array(
        [clean.l.values[0], clean.l.values[-1], clean.m.values[0], clean.m.values[-1]]
    )
    / ARCSEC_TO_RAD
)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), constrained_layout=True)
for ax, plane, title in [
    (axes[0], restored, "SKY_RESTORED_PRIMARY_BEAM_CORRECTED"),
    (axes[1], expected_restored, "model convolved with the clean beam"),
]:
    im = ax.imshow(
        plane.T,
        origin="lower",
        extent=extent_arcsec,
        cmap="viridis",
        vmin=0,
        vmax=expected_restored.max(),
    )
    ax.set_title(title, fontsize=10)
    fig.colorbar(im, ax=ax, shrink=0.8, label="Jy/beam")
im = axes[2].imshow(
    difference.T,
    origin="lower",
    extent=extent_arcsec,
    cmap="RdBu_r",
    vmin=-np.abs(difference).max(),
    vmax=np.abs(difference).max(),
)
axes[2].set_title("difference", fontsize=10)
fig.colorbar(im, ax=axes[2], shrink=0.8, label="Jy/beam")
for ax in axes:
    ax.set_xlabel("l [arcsec]")
    ax.set_ylabel("m [arcsec]")
plt.show()

# cuts through the centre along the major and minor axes
offsets = np.linspace(-2.0, 2.0, 201) * ARCSEC_TO_RAD
axis_vectors = {
    "major axis": (np.sin(position_angle), np.cos(position_angle)),
    "minor axis": (np.cos(position_angle), -np.sin(position_angle)),
}
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True, sharey=True)
for ax, (label, (e_l, e_m)) in zip(axes, axis_vectors.items(), strict=True):
    i_pix = image_size[0] // 2 + offsets * e_l / cell_size[0]
    j_pix = image_size[1] // 2 + offsets * e_m / cell_size[1]
    for plane, style, name in [
        (restored, "b-", "restored"),
        (expected_restored, "r--", "expected"),
    ]:
        from scipy.ndimage import map_coordinates

        ax.plot(
            offsets / ARCSEC_TO_RAD,
            map_coordinates(plane, [i_pix, j_pix], order=1),
            style,
            label=name,
        )
    ax.set_title(f"cut along the {label}")
    ax.set_xlabel("offset from the disk centre [arcsec]")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Jy/beam")
axes[0].legend()
plt.show()

table = flux_table(restored, difference)
table

In [ ]:
# the same checks the component test makes
np.testing.assert_allclose(
    table.loc[0, "restored"], table.loc[0, "expected"], rtol=0.02
)
np.testing.assert_allclose(
    table.loc[1, "restored"], table.loc[1, "expected"], rtol=0.02
)
assert table.loc[2, "restored"] < 0.02
print("disk flux and shape recovered")

## The same disk with the Adaptive Scale Pixel (ASP) deconvolver

Hogbom CLEAN models the sky as delta functions.  The **Adaptive Scale Pixel** deconvolver
(``deconvolver="asp"``, a port of CASA's ``AspMatrixCleaner``) instead fits one Gaussian *Aspen*
per iteration, optimising its amplitude, position and scale against the residual, and falls back
to point components where nothing extended is left.  Each iteration is therefore more expensive
(the Aspen fit is a small optimisation costing several FFTs), but the model it builds is a smooth
image rather than a set of spikes.

The cells image the same processing set with ASP and compare it with the Hogbom result.  On
this disk the first few tens of Aspens capture the bulk of the emission; the remaining
iterations, largely in the point-source fallback, chase the sharp outer edge of the
limb-darkened profile, which no Gaussian fits.  So the iteration count ends up comparable to
Hogbom's while the run takes longer, and the two restored images agree with the analytic model
equally well.  The ASP model can be compared with the input sky directly, pixel by pixel, which
makes no sense for a delta-function model.

In [ ]:
from astroviper.processing_functions.imaging.utils.iteration_control import (
    get_iterations_done_from_imaging_dict,
)

clean_asp, asp_result = image_simulation(
    "disk_sim.ps.zarr", "disk_sim_asp.img.zarr", niter=3000, deconvolver="asp"
)
restored_asp = clean_asp.SKY_RESTORED_PRIMARY_BEAM_CORRECTED.isel(
    time=0, frequency=0, polarization=0
).values
difference_asp = restored_asp - expected_restored


def deconvolution_summary(result):
    """Model update iterations (channel 0) and deconvolution wall time summed over chunks."""
    timing = result["timing_node_tasks"]
    columns = [c for c in timing.columns if "deconvolve" in c]
    return {
        "model update iterations (channel 0)": get_iterations_done_from_imaging_dict(
            result["deconvolution"], chan=0
        ),
        "deconvolution time [s]": float(timing[columns].to_numpy().sum()),
    }


summary = pd.DataFrame(
    {
        "Hogbom": deconvolution_summary(hogbom_result),
        "ASP": deconvolution_summary(asp_result),
    }
)

In [ ]:
model_hogbom = clean.SKY_MODEL.isel(time=0, frequency=0, polarization=0).values
model_asp = clean_asp.SKY_MODEL.isel(time=0, frequency=0, polarization=0).values
model_asp_error = model_asp - model  # the ASP model against the input sky (Jy/pixel)
print(
    f"model peak [Jy/pixel]: Hogbom {model_hogbom.max():.4f}, ASP {model_asp.max():.4f}, "
    f"input sky {model.max():.4f}"
)

fig, axes = plt.subplots(2, 3, figsize=(15, 8.6), constrained_layout=True)
for ax, plane, title in [
    (axes[0, 0], model_hogbom, "SKY_MODEL, Hogbom (delta components)"),
    (axes[0, 1], model_asp, "SKY_MODEL, ASP (Gaussian components)"),
]:
    im = ax.imshow(
        plane.T,
        origin="lower",
        extent=extent_arcsec,
        cmap="viridis",
        vmin=0,
        vmax=plane.max(),
    )
    ax.set_title(title, fontsize=10)
    fig.colorbar(im, ax=ax, shrink=0.8, label="Jy/pixel")
im = axes[0, 2].imshow(
    model_asp_error.T,
    origin="lower",
    extent=extent_arcsec,
    cmap="RdBu_r",
    vmin=-np.abs(model_asp_error).max(),
    vmax=np.abs(model_asp_error).max(),
)
axes[0, 2].set_title("ASP model - input sky", fontsize=10)
fig.colorbar(im, ax=axes[0, 2], shrink=0.8, label="Jy/pixel")
for ax, plane, title in [
    (axes[1, 0], restored, "SKY_RESTORED_PRIMARY_BEAM_CORRECTED, Hogbom"),
    (axes[1, 1], restored_asp, "SKY_RESTORED_PRIMARY_BEAM_CORRECTED, ASP"),
]:
    im = ax.imshow(
        plane.T,
        origin="lower",
        extent=extent_arcsec,
        cmap="viridis",
        vmin=0,
        vmax=expected_restored.max(),
    )
    ax.set_title(title, fontsize=10)
    fig.colorbar(im, ax=ax, shrink=0.8, label="Jy/beam")
im = axes[1, 2].imshow(
    difference_asp.T,
    origin="lower",
    extent=extent_arcsec,
    cmap="RdBu_r",
    vmin=-np.abs(difference_asp).max(),
    vmax=np.abs(difference_asp).max(),
)
axes[1, 2].set_title("ASP restored - expected", fontsize=10)
fig.colorbar(im, ax=axes[1, 2], shrink=0.8, label="Jy/beam")
for ax in axes.ravel():
    ax.set_xlabel("l [arcsec]")
    ax.set_ylabel("m [arcsec]")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True, sharey=True)
for ax, (label, (e_l, e_m)) in zip(axes, axis_vectors.items(), strict=True):
    i_pix = image_size[0] // 2 + offsets * e_l / cell_size[0]
    j_pix = image_size[1] // 2 + offsets * e_m / cell_size[1]
    for plane, style, name in [
        (restored, "b-", "Hogbom"),
        (restored_asp, "g-", "ASP"),
        (expected_restored, "r--", "expected"),
    ]:
        ax.plot(
            offsets / ARCSEC_TO_RAD,
            map_coordinates(plane, [i_pix, j_pix], order=1),
            style,
            label=name,
        )
    ax.set_title(f"cut along the {label}")
    ax.set_xlabel("offset from the disk centre [arcsec]")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Jy/beam")
axes[0].legend()
plt.show()

table_asp = flux_table(restored_asp, difference_asp)
comparison = pd.DataFrame(
    {
        "Hogbom": list(table.restored) + list(summary.Hogbom),
        "ASP": list(table_asp.restored) + list(summary.ASP),
        "expected": list(table.expected) + [np.nan, np.nan],
    },
    index=list(table.quantity) + list(summary.index),
).round(4)
comparison

In [ ]:
# ASP recovers the disk as well as Hogbom does (the component test checks both)
np.testing.assert_allclose(
    table_asp.loc[0, "restored"], table_asp.loc[0, "expected"], rtol=0.02
)
np.testing.assert_allclose(
    table_asp.loc[1, "restored"], table_asp.loc[1, "expected"], rtol=0.02
)
assert table_asp.loc[2, "restored"] < 0.02
print("disk flux and shape recovered with ASP")

## Clean up

In [ ]:
import shutil

for path in [
    "disk_sim.ps.zarr",
    "disk_sim_dirty.img.zarr",
    "disk_sim_clean.img.zarr",
    "disk_sim_asp.img.zarr",
]:
    shutil.rmtree(path, ignore_errors=True)
viper_client.close()